In [0]:
# instalação de openpyxl necessária para ler arquivos excel.
# após a instalação é necessário reiniciar o kernel
%pip install openpyxl

In [0]:
import os
import pandas as pd
from pyspark.sql.functions import col, count, when, lit
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

# --- CONFIGURAÇÃO ---
ROOT_PATH = "/Volumes/hackathon_2025/default/source/"

# --- DEFINIÇÃO DO SCHEMA (BLINDAGEM) ---
# Isso impede o erro [CANNOT_INFER_TYPE_FOR_FIELD]
report_schema = StructType([
    StructField("arquivo", StringType(), True),
    StructField("formato", StringType(), True),
    StructField("total_linhas", LongType(), True),
    StructField("coluna", StringType(), True),
    StructField("tipo_dado", StringType(), True),
    StructField("qtd_nulos_ou_vazios", LongType(), True),
    StructField("pct_problemas", DoubleType(), True)
])

# --- CONTROLE DE DUPLICIDADE PARQUET ---
processed_parquet_folders = set()

# --- FUNÇÃO DE ANÁLISE ---
def analyze_spark_df(df, filename, filetype):
    row_count = df.count()
    cols = df.columns
    
    null_counts = df.select([
        count(when(
            col(c).isNull() | 
            (col(c) == "") | 
            (col(c) == "NaN") |
            (col(c) == "null"), c
        )).alias(c) 
        for c in cols
    ]).collect()[0].asDict()
    
    metrics = []
    for c in cols:
        n_nulls = null_counts[c]
        pct_nulls = (n_nulls / row_count * 100) if row_count > 0 else 0
        
        metrics.append({
            "arquivo": str(filename),
            "formato": str(filetype),
            "total_linhas": int(row_count), # Força int Python
            "coluna": str(c),
            "tipo_dado": "STRING (Raw)",
            "qtd_nulos_ou_vazios": int(n_nulls), # Força int Python
            "pct_problemas": float(round(pct_nulls, 2)) # Força float Python
        })
    return metrics

# --- EXECUÇÃO PRINCIPAL ---
all_results = []

print(f"🚀 Iniciando varredura COM SCHEMA EXPLÍCITO em: {ROOT_PATH}\n")

for root, dirs, files in os.walk(ROOT_PATH):
    for file in files:
        full_path = os.path.join(root, file)
        rel_path = os.path.relpath(full_path, ROOT_PATH)
        
        try:
            # 1. PARQUET (Lógica de Agrupamento)
            if file.endswith(".parquet"):
                parquet_folder = os.path.dirname(full_path)
                
                if parquet_folder in processed_parquet_folders:
                    continue
                
                rel_folder_path = os.path.relpath(parquet_folder, ROOT_PATH)
                print(f"📂 Processando Dataset Parquet: {rel_folder_path}")
                
                df = spark.read.parquet(parquet_folder)
                all_results.extend(analyze_spark_df(df, rel_folder_path, "PARQUET (DATASET)"))
                
                processed_parquet_folders.add(parquet_folder)
                
            # 2. CSV
            elif file.lower().endswith(".csv"):
                print(f"📄 Processando CSV: {rel_path}")
                try:
                    df = spark.read.option("header", "true") \
                                   .option("inferSchema", "false") \
                                   .option("delimiter", ";") \
                                   .csv(full_path)
                    if len(df.columns) <= 1:
                        df = spark.read.option("header", "true") \
                                       .option("inferSchema", "false") \
                                       .option("delimiter", ",") \
                                       .csv(full_path)
                except:
                    continue
                all_results.extend(analyze_spark_df(df, rel_path, "CSV"))
                
            # 3. EXCEL
            elif file.lower().endswith(".xlsx") or file.lower().endswith(".xls"):
                print(f"📊 Processando Excel: {rel_path}")
                try:
                    xls_file = pd.ExcelFile(full_path)
                    for sheet_name in xls_file.sheet_names:
                        # Lê como string para evitar NaN numérico do Pandas
                        pdf = pd.read_excel(xls_file, sheet_name=sheet_name, dtype=str)
                        p_rows = len(pdf)
                        file_identifier = f"{rel_path} [Aba: {sheet_name}]"
                        
                        if p_rows == 0: continue

                        for c in pdf.columns:
                            # Contagem no Pandas
                            n_nulls = pdf[c].isnull().sum() + (pdf[c] == "").sum() + (pdf[c] == "nan").sum()
                            pct = (n_nulls / p_rows * 100) if p_rows > 0 else 0
                            
                            all_results.append({
                                "arquivo": str(file_identifier),
                                "formato": "EXCEL",
                                "total_linhas": int(p_rows), # Remove tipo numpy
                                "coluna": str(c),
                                "tipo_dado": "STRING (Raw)",
                                "qtd_nulos_ou_vazios": int(n_nulls), # Remove tipo numpy
                                "pct_problemas": float(round(pct, 2)) # Remove tipo numpy
                            })
                except Exception as e:
                    print(f"Erro Excel: {e}")

        except Exception as e:
            print(f"Erro genérico: {e}")

# --- RELATÓRIO FINAL ---
if all_results:
    print("\n✅ Análise Concluída! Gerando tabela...")
    # AQUI ESTÁ A MÁGICA: Passamos o schema=report_schema
    final_df = spark.createDataFrame(all_results, schema=report_schema)
    display(final_df.orderBy(col("pct_problemas").desc()))
else:
    print("⚠️ Nada encontrado.")


In [0]:
# --- 1. SALVAR A TABELA DETALHADA (Melhor Prática) ---
target_table = "hackathon_2025.default.relatorio_qualidade_dados"

print(f"💾 Salvando dados detalhados em: {target_table} ...")

# mode("overwrite") garante que se você rodar de novo, ele atualiza a tabela
final_df.write.mode("overwrite").saveAsTable(target_table)

print("✅ Tabela salva com sucesso!")